# Week 2 — LLM 기반 수면 자세 분류
**환경**: Python 3.10 / VSCode Jupyter

**동작**
- week2_01 / week2_02에서 Supabase Storage에 업로드된 수면 이미지를 가져옴
- 각 이미지를 Claude Vision API에 전송해 수면 자세 추론
- 분류 결과(Supine / Lateral_L / Lateral_R / Prone / Unknown)를 Supabase `posture_log`에 저장

**실행 순서**: 셀을 위에서부터 순서대로 실행하세요 (Shift+Enter)

## 0. 패키지 설치

## 0. 패키지 설치
처음 한 번만 실행하면 됩니다

In [1]:
%pip install google-generativeai Pillow opencv-python numpy pandas matplotlib

  Using cached protobuf-7.34.1-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 1.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 2.2 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 2.1 MB/s  0:00:07 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [google-generativeai]ogle-ai-generativelanguage]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import io
import requests
import google.generativeai as genai
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from datetime import datetime
from pathlib import Path

sys.path.insert(0, "..")     # backend/ → utils 패키지 접근용
sys.path.insert(0, "../..")  # 프로젝트 루트 → config.py 접근용

from utils.db import list_storage_images, insert_posture, fetch_posture_by_date

print("[OK] 라이브러리 로드 완료")

/opt/miniconda3/envs/my_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/var/folders/td/3yfbdgyx5hq85zjwbzb4gd8m0000gn/T/ipykernel_44108/1751643088.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


[OK] 라이브러리 로드 완료


## 1. 설정값

In [2]:
# =============================================
#  설정값 — 필요 시 여기서 수정
# =============================================

GEMINI_API_KEY = "여기에_API_키_입력"   # https://aistudio.google.com/app/apikey

# 분류할 날짜 (Storage 폴더명 기준, 오늘 날짜 자동 설정)
TARGET_DATE = datetime.now().strftime("%Y-%m-%d")   # ex) "2026-05-11"

# 사용할 Gemini 모델
MODEL = "gemini-1.5-flash"   # 빠르고 저렴 / 더 정확하게: gemini-1.5-pro

print(f"분류 날짜 : {TARGET_DATE}")
print(f"사용 모델 : {MODEL}")

분류 날짜 : 2026-05-11
사용 모델 : gemini-1.5-flash


## 2. LLM 자세 분류 함수 정의

Claude Vision API에 이미지를 보내 수면 자세를 추론합니다.

**분류 라벨**
| 라벨 | 의미 |
|------|------|
| `Supine` | 천장을 보고 누운 자세 (앙와위) |
| `Lateral_L` | 왼쪽으로 누운 자세 |
| `Lateral_R` | 오른쪽으로 누운 자세 |
| `Prone` | 엎드려 누운 자세 |
| `Unknown` | 판별 불가 (사람이 없거나 이미지 불량) |

In [3]:
POSTURE_LABELS = ["Supine", "Lateral_L", "Lateral_R", "Prone", "Unknown"]

SYSTEM_PROMPT = """\
당신은 수면 자세 분류 전문가입니다.
제공된 이미지에서 침대 위 사람의 수면 자세를 다음 중 하나로 분류하세요.

- Supine    : 천장을 보고 누운 자세 (앙와위)
- Lateral_L : 왼쪽으로 눕고 오른쪽이 위로 오는 자세
- Lateral_R : 오른쪽으로 눕고 왼쪽이 위로 오는 자세
- Prone     : 엎드려 누운 자세 (복와위)
- Unknown   : 사람이 보이지 않거나 판별 불가

반드시 위 5개 라벨 중 하나만 반환하세요. 다른 텍스트는 포함하지 마세요.\
"""

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel(
    model_name=MODEL,
    system_instruction=SYSTEM_PROMPT,
)


def classify_with_llm(image_url: str) -> str:
    """
    이미지 URL 또는 로컬 경로를 Gemini Vision에 전송해 수면 자세를 분류
    반환: 자세 라벨 (str)
    """
    try:
        if image_url.startswith("http"):
            resp = requests.get(image_url, timeout=15)
            if resp.status_code != 200:
                return "Unknown"
            image_bytes = resp.content
        else:
            with open(image_url, "rb") as f:
                image_bytes = f.read()
    except Exception as e:
        print(f"  [이미지 로드 실패] {e}")
        return "Unknown"

    try:
        pil_image = Image.open(io.BytesIO(image_bytes))
        response  = model.generate_content(
            ["이 이미지의 수면 자세를 분류하세요.", pil_image]
        )
        raw = response.text.strip()
        for label in POSTURE_LABELS:
            if label.lower() in raw.lower():
                return label
        return "Unknown"

    except Exception as e:
        print(f"  [LLM 오류] {e}")
        return "Unknown"


print("[OK] Gemini 분류 함수 정의 완료")

[OK] Gemini 분류 함수 정의 완료


## 3. Supabase Storage에서 이미지 목록 가져오기

In [4]:
date_folder = TARGET_DATE.replace("-", "")   # "2026-05-11" → "20260511"
images      = list_storage_images(date_folder)

regular_imgs = [img for img in images if img["capture_type"] == "regular"]
motion_imgs  = [img for img in images if img["capture_type"] == "motion"]

print(f"Storage [{date_folder}] 이미지: 총 {len(images)}장")
print(f"  정기 촬영  : {len(regular_imgs)}장")
print(f"  움직임 감지: {len(motion_imgs)}장")

if not images:
    print("\n이미지가 없습니다. week2_01 또는 week2_02를 먼저 실행해서 이미지를 수집하세요.")

Storage [20260511] 이미지: 총 0장
  정기 촬영  : 0장
  움직임 감지: 0장

이미지가 없습니다. week2_01 또는 week2_02를 먼저 실행해서 이미지를 수집하세요.


## 4. 단일 이미지 테스트 (선택)
전체 배치 실행 전에 1장만 먼저 테스트합니다

In [5]:
if not images:
    print("이미지 없음 — 3번 셀을 먼저 실행하세요")
else:
    sample = images[0]
    print(f"테스트 이미지: {sample['name']}")
    print(f"URL: {sample['url']}\n")

    result = classify_with_llm(sample["url"])
    print(f"LLM 분류 결과: {result}")

    # 이미지 미리보기
    import cv2
    resp = requests.get(sample["url"], timeout=15)
    arr  = np.frombuffer(resp.content, np.uint8)
    img  = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8, 5))
    plt.imshow(cv2.resize(img, (640, 360)))
    plt.title(f"LLM 분류: {result}", fontsize=13)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

이미지 없음 — 3번 셀을 먼저 실행하세요


## 5. 전체 이미지 배치 분류 → Supabase 저장
모든 이미지를 순서대로 LLM에 전송하고 결과를 `posture_log`에 저장합니다

In [6]:
if not images:
    print("이미지 없음 — 3번 셀을 먼저 실행하세요")
else:
    print(f"LLM 자세 분류 시작 — 총 {len(images)}장\n")
    batch_results = []
    errors        = 0

    for i, img_info in enumerate(images):
        posture = classify_with_llm(img_info["url"])

        # posture_log에 저장 (이미지는 이미 Storage에 있으므로 upload=False)
        insert_posture(
            timestamp    = img_info["timestamp"],
            posture      = posture,
            angle        = 0.0,
            capture_type = img_info["capture_type"],
            image_path   = img_info["url"],
            upload       = False,
        )

        batch_results.append({
            "timestamp":    img_info["timestamp"],
            "datetime":     datetime.fromtimestamp(img_info["timestamp"]).strftime("%H:%M:%S"),
            "capture_type": img_info["capture_type"],
            "posture":      posture,
        })

        status = "✅" if posture != "Unknown" else "❓"
        print(f"  {status} [{batch_results[-1]['datetime']}] "
              f"{img_info['capture_type']:8s} → {posture}")

        if posture == "Unknown":
            errors += 1

    print(f"\n완료! 총 {len(batch_results)}장 분류")
    print(f"  성공: {len(batch_results) - errors}장 / 불명: {errors}장")

이미지 없음 — 3번 셀을 먼저 실행하세요


## 6. 분류 결과 확인 & 시각화

In [8]:
rows = fetch_posture_by_date(TARGET_DATE)

if not rows:
    print("결과 없음 — 5번 셀을 먼저 실행하세요")
else:
    df = pd.DataFrame(rows)
    print(f"총 기록 수: {len(df)}개\n")
    print(df[["datetime", "capture_type", "posture"]].tail(10).to_string(index=False))

    print("\n── 자세별 집계 ──")
    counts = df["posture"].value_counts()
    for posture, cnt in counts.items():
        print(f"  {posture:12s}: {cnt}회 ({cnt/len(df)*100:.1f}%)")

총 기록 수: 8개

                 datetime capture_type posture
2026-05-11T09:22:52+00:00      regular  Supine
2026-05-11T09:30:15+00:00      regular  Supine
2026-05-11T09:35:04+00:00      regular  Supine
2026-05-11T09:37:37+00:00      regular  Supine
2026-05-11T09:38:10+00:00      regular  Supine
2026-05-11T09:40:14+00:00      regular  Supine
2026-05-11T09:42:47+00:00      regular  Supine
2026-05-11T09:45:26+00:00      regular  Supine

── 자세별 집계 ──
  Supine      : 8회 (100.0%)


In [7]:
if not rows:
    print("결과 없음 — 5번 셀을 먼저 실행하세요")
else:
    COLORS = {
        "Supine":    "#5BBF72",
        "Lateral_L": "#F4A742",
        "Lateral_R": "#E07BB5",
        "Prone":     "#5B8EBF",
        "Unknown":   "#AAAAAA",
    }

    counts = df["posture"].value_counts()
    colors = [COLORS.get(p, "#CCCCCC") for p in counts.index]

    posture_order = ["Supine", "Lateral_L", "Lateral_R", "Prone", "Unknown"]
    posture_num   = {p: i for i, p in enumerate(posture_order)}
    df["posture_num"]   = df["posture"].map(posture_num)
    scatter_colors      = [COLORS.get(p, "#CCCCCC") for p in df["posture"]]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # 파이차트
    axes[0].pie(counts.values, labels=counts.index,
                colors=colors, autopct="%1.1f%%",
                startangle=140, pctdistance=0.82)
    axes[0].set_title(f"수면 자세 분포 ({TARGET_DATE})", fontsize=13)

    # 시간대별 자세 변화
    axes[1].scatter(range(len(df)), df["posture_num"],
                    c=scatter_colors, s=80, zorder=3)
    axes[1].plot(range(len(df)), df["posture_num"],
                 color="#CCCCCC", linewidth=0.8, zorder=2)
    axes[1].set_yticks(range(len(posture_order)))
    axes[1].set_yticklabels(posture_order)
    axes[1].set_xlabel("촬영 순서")
    axes[1].set_title("시간대별 자세 변화", fontsize=13)
    axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

NameError: name 'rows' is not defined